# Covariance Estimation, Shrinkage y Portfolio Robustness

> Esta semana vamos a profundizar en un problema directamente relacionado con el estudio de Portfolio Construction de Anton Vorobets, los modelos de factores de QuantFrame y el Quant Desk:

---
$  \boxed{\text{Una matriz de covarianza estimada no es la verdadera matriz de riesgo.}} $
Hasta ahora hemos trabajado con matrices $ \Sigma $  conocidas. 

En un problema real, tenemos que estimarlas a partir de datos históricos.

Esto introduce un riesgo importante: un optimizador puede explotar errores de estimación y construir un portafolio aparentemente óptimo, pero muy frágil fuera de muestra.

--- 

Hoy estudiaremos ese problema desde tres perspectivas: matemática, computacional y práctica.

| Día | Tema | Competencia |
|---|---|---|
| **Lunes 12** | Covariance estimation y shrinkage | Matemáticas + estadística + portfolio |
| Martes 13 | Factor risk vs idiosyncratic risk | Modelos factoriales + Python |
| Miércoles 14 | Order book, spread y adverse selection | Mercados + microestructura |
| Jueves 15 | Entropy Pooling y views inconsistentes | Probabilidad + portfolio/risk |
| Viernes 16 | Lectura crítica de research: validación de estrategias | Investigación cuantitativa |
| Sábado 17 | Quant Desk: integración y decisión de riesgos | Aplicación práctica |
| Domingo 18 | **Descanso** | Sin actividades |

<b> Estadística |¿Puede Shrinkage superar a un optimizador aparentemente más preciso? </b>

Vamos a simular ocho activos con una fuente común de riesgo y componentes idiosincráticos.

```python
import numpy as np

rng = np.random.default_rng(117)
n_assets = 8
T_train = 50
T_test = 252
T = T_train + T_test

common_factor = rng.normal(0, 0.007, size=(T, 1))
idio_noise = rng.normal(0, 0.009, size=(T, n_assets))
returns = common_factor + idio_noise

train = returns[:T_train]
test = returns[T_train:]
```

La muestra de entrenamiento es deliberadamente pequeña.

**Parte A — Covarianza muestral**

Calcula:

$\hat{\Sigma}_{sample} = \text{Cov}(R_{train})$.

Después construye una matriz regularizada:

$\hat{\Sigma}_{shrink} = (1 - \lambda)\hat{\Sigma}_{sample} + \lambda D$

donde:

$D = \text{diag}(\hat{\Sigma}_{sample})$

y:

$\lambda = 0.50.$

Este shrinkage reduce las covarianzas fuera de la diagonal sin modificar las varianzas individuales.

No es necesario estimar un $\lambda$ óptimo. Hoy lo fijamos antes de ver Test.

**Parte B — Portfolio Optimization**

Construye tres portafolios:

**Portfolio A: Sample Minimum Variance**

$w_A = \frac{\hat{\Sigma}_{sample}^{-1} \mathbf{1}}{\mathbf{1}^T \hat{\Sigma}_{sample}^{-1} \mathbf{1}}.$

**Portfolio B: Shrinkage Minimum Variance**

$ w_B = \frac{\hat{\Sigma}_{shrink}^{-1} \mathbf{1}}{\mathbf{1}^T \hat{\Sigma}_{shrink}^{-1} \mathbf{1}} $

**Portfolio C: Equal Weight**

$ w_C = [1/8 \quad 1/8 \quad \dots \quad 1/8]^T $

Utiliza `np.linalg.solve()` en lugar de calcular explícitamente las matrices inversas.

**Parte C — Evaluación OOS**

Evalúa los tres portafolios exclusivamente sobre `test`.

Calcula:

$\sigma_{OOS} = \text{Std}(R_{test}w)\sqrt{252}.$

También calcula:

$\text{GrossExposure} = \sum_{i} |w_i|.$

Y:

$\text{MaxWeight} = \max_{i} |w_i|.$


Construye:

| Portfolio | OOS Vol | Gross Exposure | Max Absolute Weight |
|---|---:|---:|---:|
| Sample MinVar | | | |
| Shrinkage MinVar | | | |
| Equal Weight | | | |

Preguntas:

¿Por qué la matriz muestral puede generar posiciones extremas aunque sus estimaciones sean estadísticamente razonables?

> <b> RESPUESTA </b>

> • El optimizador de Markowitz es un "detector de anomalías". Si una acción tuvo una covarianza ligeramente baja por pura suerte o ruido en la muestra histórica, el optimizador asumirá que descubrió una mina de oro de diversificación. Para explotarla al máximo, asignará un peso extremo positivo a ese activo y, para compensar el riesgo, un peso extremo negativo (venta en corto) al activo con el que se parece.
• Aunque tus estimaciones históricas sean promedios estadísticamente válidos, contienen ruido. La inversión de la matriz muestral magnifica ese ruido, transformándolo en ponderaciones absurdas ej. +400% en un activo y -300% en otro que son imposibles de ejecutar en la vida real.


¿Por qué shrinkage introduce sesgo deliberadamente?

> <b> RESPUESTA </b> 

> • La matriz de covarianza de la muestra es un estimador insesgado (en promedio, con infinitos datos, daría la covarianza real), pero tiene una varianza altísima (cambia drásticamente si quitas o pones un par de días de datos).

> Shrinkage toma la matriz muestral y la "empuja" (o encoge) hacia una estructura matemática rígidamente estructurada, usualmente la matriz de varianzas iguales y covarianzas cero (el equivalente al portafolio Equal Weight). Esta matriz objetivo es sumamente estable, pero está sesgada (asume falsamente que ninguna acción está correlacionada con otra).

> Al mezclar ambas matrices (\(\Sigma_{shrink} = \delta F + (1-\delta)\hat{\Sigma}\)), estás "contaminando" a propósito tus datos reales con una suposición teórica falsa (el objetivo). Introduces sesgo porque estás ignorando parte de la información histórica real a cambio de ganar estabilidad numérica. La matriz resultante ya no es perfectamente insesgada, pero es infinitamente más fácil de invertir y no explota matemáticamente.

¿Bajo qué condiciones puede compensarse ese sesgo mediante menor error fuera de muestra?

> <b> RESPUESTA </b>

> El sesgo se compensa exitosamente cuando la reducción en la varianza del error es mayor que el tamaño del sesgo introducido. Esto da como resultado un Error Cuadrático Medio (MSE) más bajo fuera de la muestra.

1. • Muestras Pequeñas o Datos Insuficientes N  grande, T pequeño: Si estás optimizando un portafolio con muchas acciones (N pero tienes pocos días de historial T, la matriz muestral está llena de ruido puro. El sesgo de shrinkage actúa como un escudo que filtra ese ruido.

2. • Alta Correlación entre Activos: Si en tu universo del Reto Actinver tienes muchas acciones del mismo sector (ej. financiero o tecnológico), la matriz muestral colapsará al invertirse. El sesgo de shrinkage "separa" artificialmente los activos, permitiendo que el optimizador funcione sin dar posiciones extremas.

3. • Inestabilidad Estructural (Cambios de Régimen): El pasado nunca se repite exactamente igual en el futuro. Si optimizas con una matriz muestral perfecta del mes pasado, fallará fuera de muestra porque el mercado cambia. Al introducir el sesgo de shrinkage, estás creando un portafolio más "conservador" y robusto que tolera mucho mejor los cambios y sorpresas del mercado real (Out-of-Sample).

**Advertencia importante:** La fórmula utilizada permite posiciones cortas. Si obtienes pesos negativos, no debes interpretar automáticamente el portfolio como ejecutable en un entorno long-only.

Además, una sola simulación no demuestra que shrinkage sea superior. Si te quedan cinco minutos, repite el experimento con 20 semillas y compara la mediana de volatilidad OOS.

**Entregable:** Un DataFrame y tres conclusiones breves.


In [5]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(117)
n_assets = 8
T_train = 50
T_test = 252
T = T_train + T_test

common_factor = rng.normal(0, 0.007, size=(T, 1))
idio_noise = rng.normal(0, 0.009, size=(T, n_assets))
returns = common_factor + idio_noise

train = returns[:T_train]
test = returns[T_train:]
# PARTE A - Covarianza Muestral

# calculando la matriz de cov 
cov_sample = np.cov(train, rowvar = False)
# matriz regularizada shrink
lambda_ = 0.50 
D = np.diag(np.diag(cov_sample))
cov_shrink = ( (1 - lambda_)*cov_sample + lambda_*D)

# PARTE B - optimización de portafolio 

# Portafolio A - Sample Minumum variance

# vector de unos debe tener el mismo tamaño que el numero de activos 
num_activos = cov_sample.shape[0]
unos = np.ones(num_activos)
# matriz inversa x vectores de unos  : Usar np.linalg.solve 
numerador = np.linalg.solve(cov_sample, unos)
# Suma total de los elementos de ese vector resultante
denominador = np.sum(numerador)
# Obtenemos los pesos óptimos del Portafolio de Mínima Varianza
pesos_optimos = numerador / denominador
print("Pesos del portafolio:", pesos_optimos)

# Portafolio B - Shrinkage minima varianza 
num_activos_sh = cov_shrink.shape[0]
unos_sh = np.ones(num_activos_sh)
numerador_sh = np.linalg.solve(cov_shrink, unos_sh)
denominador_sh = np.sum(numerador_sh)
pesos_optimos_sh = numerador_sh / denominador_sh
print("Pesos del portafolio Shrinkage:", pesos_optimos_sh)

# Portafolio C : Equal Weight 
# vector de 8 posiciones, cada una con el valor 1/8 (0.125)
pesos_c = np.full(num_activos, 1 / num_activos)
print("Pesos del portafolio Iguales:", pesos_c)

# Evaluacion OSS

def metricas_oss(weights): 
    portfolio_returns = test @ weights
    sigma_oss = (np.std(portfolio_returns, ddof= 1)* np.sqrt(252) )
    gross_exposure = np.sum(np.abs(weights))
    max_weight = np.max(np.abs(weights))
    return {'sigma_oss': sigma_oss,
            'gross_exposure': gross_exposure, 
            'max_weight': max_weight}

mean_variance = metricas_oss(pesos_optimos)
shrink_ = metricas_oss(pesos_optimos_sh)
pesos_iguales = metricas_oss(pesos_c)
nombres = ['Sample_MinVar', 'Shrinkage_MinVar', 'Equal_weight']

df =  pd.DataFrame({
    'Volatilidad_OOS': [mean_variance['sigma_oss'], shrink_['sigma_oss'], pesos_iguales['sigma_oss']],
    'Exposicion_Bruta': [mean_variance['gross_exposure'], shrink_['gross_exposure'], pesos_iguales['gross_exposure']],
    'Peso_Maximo': [mean_variance['max_weight'], shrink_['max_weight'], pesos_iguales['max_weight']]
}, index=nombres)
df

Pesos del portafolio: [ 0.25383629  0.25847044  0.19191596 -0.02063066  0.16499304 -0.04238504
  0.02099527  0.17280471]
Pesos del portafolio Shrinkage: [0.20414358 0.21099763 0.14357483 0.03697444 0.14014784 0.03307953
 0.07344018 0.15764197]
Pesos del portafolio Iguales: [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]


,Volatilidad_OOS,Exposicion_Bruta,Peso_Maximo
Sample_MinVar,0.129133,1.126031,0.258470
Shrinkage_MinVar,0.125453,1.000000,0.210998
Equal_weight,0.123542,1.000000,0.125000


# <b> Martes </B>

## Factor Models & Portfolio Risk

### Tema: Riesgo sistemático, riesgo idiosincrático y concentración oculta
Ayer estudiamos cómo el error al estimar una matriz de covarianza puede producir portafolios frágiles. 

Hoy continuaremos con una pregunta más profunda:

¿De dónde proviene realmente el riesgo de nuestro portafolio?
Un portafolio puede contener muchos activos y, aun así, estar dominado por una única fuente de riesgo.

Trabajaremos con un modelo factorial:

$ \boxed{r_t=Bf_t+\varepsilon_t} $ 

donde $ B $  contiene las exposiciones a factores, $ f_t $  sus retornos y $\varepsilon_t$  los componentes residuales.

---

<b> Estadística </b>

Un cambio en la correlación entre factores altera el riesgo total


Utiliza los datos del primer ejercicio.

```python
import numpy as np

w = np.array([0.50, 0.30, 0.20])

B = np.array([
    [1.2,  0.2],
    [0.8, -0.1],
    [0.2,  1.0]
])

factor_vol = np.array([0.01, 0.008])
idio_vol = np.array([0.006, 0.008, 0.010])

D = np.diag(idio_vol**2)
```
Ahora estudiamos dos regímenes:

$\rho_{normal} = 0.10$

$\rho_{stress} = 0.75.$

Para cada uno construye:

$\Sigma_f = \begin{bmatrix} \sigma_M^2 & \rho \sigma_M \sigma_{Mom} \\ \rho \sigma_M \sigma_{Mom} & \sigma_{Mom}^2 \end{bmatrix}.$

Y posteriormente:

$\Sigma = B \Sigma_f B^\top + D.$

**Parte A — Riesgo total**

Calcula para ambos regímenes:

$\sigma_P = \sqrt{w^\top \Sigma w}.$

Obtén también el porcentaje de varianza explicado por factores.

**Parte B — Risk Contributions**

Calcula la contribución de cada activo a la varianza:

$RC_i^{var} = w_i(\Sigma w)_i.$

Después normaliza:

$RC_i^{\%} = \frac{w_i(\Sigma w)_i}{w^\top \Sigma w}.$

Comprueba que:

$\sum_{i} RC_i^{\%} = 1.$

Pregunta importante:

¿Por qué el activo con mayor peso no necesariamente tiene la mayor contribución al riesgo?

Relaciona tu explicación con la volatilidad individual y las covarianzas.

**Parte C — Interpretación**

Has observado que el riesgo aumenta al cambiar la correlación entre factores.

Sin embargo, las exposiciones $B$ permanecieron constantes.

Responde:

¿Qué cambió realmente: las decisiones de inversión, la estructura de dependencias del mercado o ambas?

En este experimento únicamente cambió la dependencia entre factores.

Entregable: Un DataFrame comparativo y las contribuciones porcentuales al riesgo por activo.



| Métrica | Normal | Stress |
|---|---:|---:|
| Factor correlation | 0.10 | 0.75 |
| Portfolio Daily Vol | | |
| Annualized Vol | | |
| Factor Risk Share | | |
| Idiosyncratic Risk Share | | |


Estadística
¿Aumenta la selección adversa cuando cambia el régimen de mercado?
Modalidad: Código.
Ahora analizaremos doce operaciones ficticias.
Para simplificar, todas ocurren con un midprice inicial de $100 y un half-spread de $0.02.
Disponemos de dos regímenes: NORMAL y FAST.


```python
import numpy as np
import pandas as pd

direction = np.array([
    1, -1,  1,  1, -1, -1,
    1, -1,  1, -1,  1, -1
])

mid_t = np.full(12, 100.0)

mid_change_5m = np.array([
    0.00, -0.01,  0.01,
    0.00, -0.02,  0.01,
    0.05, -0.06,  0.04,
   -0.03,  0.06, -0.05
])

regime = np.array(
    ["NORMAL"] * 6 + ["FAST"] * 6
)
```

```markdown
Calcula:

\[P_{exec} = M_t + d(0.02)\]

y:

\[M_{t+5} = M_t + \Delta M_{t+5}.\]

Después implementa las tres métricas del ejercicio anterior:

\[ES_t, \quad RS_t, \quad PI_t.\]

**Parte A — Diagnóstico**

Construye:
```
| Régimen | Mean ES | Mean RS | Mean PI | % RS negativo |
|---|---:|---:|---:|---:|
| NORMAL | | | | |
| FAST | | | | |

```markdown
Comprueba para cada observación:

\(ES_t - RS_t - PI_t \approx 0.\)

La diferencia debería ser prácticamente cero salvo error numérico.

**Parte B — Interpretación estadística**

Responde:

1. ¿Qué diferencia observas entre el Realized Spread de ambos regímenes?

2. ¿Qué sugiere un aumento de movimientos posteriores alineados con la dirección de las operaciones agresoras?

3. ¿Puedes afirmar que el régimen FAST causa mayor adverse selection?
```

```markdown
La tercera respuesta requiere cuidado. Tenemos únicamente seis operaciones por régimen y un conjunto de datos sintético.

No tenemos suficiente evidencia para establecer causalidad ni para generalizar el resultado.

**Parte C — Aplicación económica**

Supón que quieres desarrollar una estrategia de market making.

Tu modelo predice ingresos por spread, pero ignora completamente:

\(E[\Delta M_{t+5} \mid d_t, \mathcal{I}_t].\)

Explica qué componente de la rentabilidad podría estar sobreestimando.

**Entregable:** Un DataFrame comparativo y tres observaciones estadísticas.
```



**Research + Quant Desk**

¿Una orden límite siempre es mejor que una orden a mercado?
**Modalidad:** Análisis escrito.

Recibes el siguiente informe ficticio de un equipo de Execution Research.

**Research Memo — Passive vs Aggressive Execution**

Se evaluaron dos políticas para ejecutar señales de compra sobre un mismo universo de activos.

Todas las señales corresponden al mismo tamaño de orden y horizonte de inversión.

**Policy A — Market Orders**

| Métrica | Resultado |
|---|---:|
| Señales | 100 |
| Órdenes ejecutadas | 100 |
| Alpha bruto esperado por señal | 6 bps |
| Coste medio de ejecución | 3 bps |
| Alpha neto por operación ejecutada | 3 bps |